In [5]:
#!/usr/bin/env python3
"""
Paste this entire file into one Colab code cell and run it.
Edit only the INPUT / OUTPUT PATHS below if your filenames differ.
Requires numpy and pandas (available in Colab). No prior CI results are needed.
The original perfect-mixing ledger and source-origin extension run together.
Results and a ZIP archive are saved under /content by default.
U/D/W/N retain the accounting definitions of the standalone version:
U electrolysis-origin intensity; D downstream-origin intensity;
W labeled sales composition using total pool CI; N numerical intensity.
W also contains the numerical channel's composition contribution.
These terms are accounting attributions, not independent causal effects.
Existing output files and the ZIP at the configured paths are overwritten.
"""
import zipfile
import hashlib
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd

# ===================== INPUT / OUTPUT PATHS =====================
# Upload the three CSVs to Colab's Files panel, then edit these paths.
M0_FILE = Path("/content/m0.csv")
M2_FILE = Path("/content/m2-upramp soft0.5.csv")
SIGNAL_FILE = Path("/content/grid_mef_upramp_hourly_2024.csv")
OUTPUT_DIR = Path("/content/SETA_R3C4_results")
ZIP_FILE = Path("/content/SETA_R3C4_results.zip")
# ===============================================================


# BEGIN unmodified original constants and input/reconstruction functions
M_H2_TO_NH3  = 0.18

M_H2_TO_MEOH = 0.22

K_H2_OUT     = 0.12024

K_NH3_REC    = 0.11848

K_H2_FUEL    = 0.035294

DE_NH3  = 0.00519

DE_MEOH = 0.08

TOL = 1e-6

REQUIRED_MODEL_COLS = [

    "model_tag", "datetime_utc", "day_of_year", "hour_in_day",



    "P_elz_mw", "P_nh3_mw", "P_meoh_mw", "P_crk_mw",



    "elz_power_green_share",

    "nh3_power_green_share",

    "meoh_power_green_share",

    "crk_power_green_share",



    "grid_import_mw",

    "res_used_mw",

    "bat_ch_mw",

    "bat_dis_mw",



    "H2_prod_g_kgph", "H2_prod_ng_kgph",

    "NH3_prod_g_kgph", "NH3_prod_ng_kgph",

    "MeOH_prod_g_kgph", "MeOH_prod_ng_kgph",



    "crk_nh3_in_kgph", "crk_h2_out_kgph", "crk_h2_fuel_kgph",



    "H2_sell_g_kgph", "H2_sell_ng_kgph",

    "NH3_sell_g_kgph", "NH3_sell_ng_kgph",

    "MeOH_sell_g_kgph", "MeOH_sell_ng_kgph",



    "tank_H2_g_kg", "tank_H2_ng_kg",

    "tank_NH3_g_kg", "tank_NH3_ng_kg",

    "tank_MeOH_g_kg", "tank_MeOH_ng_kg",

]

def clip_small_negative(series, tol=TOL):

    x = series.copy()

    x[(x < 0) & (x > -tol)] = 0.0

    return x

def load_model_df(path: Path) -> pd.DataFrame:

    if not path.exists():

        raise FileNotFoundError(f"Missing model file: {path}")

    df = pd.read_csv(path)

    missing = [c for c in REQUIRED_MODEL_COLS if c not in df.columns]

    if missing:

        raise KeyError(f"{path.name} missing columns: {missing}")

    df = df.copy()

    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)

    df = df.sort_values("datetime_utc").reset_index(drop=True)

    return df

def load_signal_df(path: Path, ef_col: str) -> pd.DataFrame:

    if not path.exists():

        raise FileNotFoundError(f"Missing signal file: {path}")

    df = pd.read_csv(path)

    if "datetime_utc" not in df.columns or ef_col not in df.columns:

        raise KeyError(f"{path.name} must contain datetime_utc and {ef_col}")

    df = df[["datetime_utc", ef_col]].copy()

    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)

    df = df.sort_values("datetime_utc").drop_duplicates(subset=["datetime_utc"])

    df = df.rename(columns={ef_col: "ef_kg_per_kwh"})

    return df

def reconstruct_unit_grid_battery_split(df: pd.DataFrame, tol=TOL):

    d = df.copy()



    d["P_elz_total_mw"] = d["P_elz_mw"]

    d["P_nh3_total_mw"] = d["P_nh3_mw"]

    d["P_meoh_total_mw"] = d["P_meoh_mw"]

    d["P_crk_total_mw"] = d["P_crk_mw"]



    for col in ["elz_power_green_share", "nh3_power_green_share", "meoh_power_green_share", "crk_power_green_share"]:

        d[col] = pd.to_numeric(d[col], errors="coerce").clip(lower=0.0, upper=1.0).fillna(0.0)



    d["P_elz_green_mw"] = d["P_elz_total_mw"] * d["elz_power_green_share"]

    d["P_nh3_green_mw"] = d["P_nh3_total_mw"] * d["nh3_power_green_share"]

    d["P_meoh_green_mw"] = d["P_meoh_total_mw"] * d["meoh_power_green_share"]

    d["P_crk_green_mw"] = d["P_crk_total_mw"] * d["crk_power_green_share"]



    d["P_elz_ng_mw"] = d["P_elz_total_mw"] - d["P_elz_green_mw"]

    d["P_nh3_ng_mw"] = d["P_nh3_total_mw"] - d["P_nh3_green_mw"]

    d["P_meoh_ng_mw"] = d["P_meoh_total_mw"] - d["P_meoh_green_mw"]

    d["P_crk_ng_mw"] = d["P_crk_total_mw"] - d["P_crk_green_mw"]



    for col in ["P_elz_ng_mw", "P_nh3_ng_mw", "P_meoh_ng_mw", "P_crk_ng_mw"]:

        d[col] = clip_small_negative(d[col], tol=tol)



    d["P_ng_sum_units_mw"] = (

        d["P_elz_ng_mw"] +

        d["P_nh3_ng_mw"] +

        d["P_meoh_ng_mw"] +

        d["P_crk_ng_mw"]

    )



    def alloc_bat(unit_ng, ng_sum, bat_dis):

        return np.where(

            (ng_sum > tol) & (bat_dis > tol),

            bat_dis * unit_ng / ng_sum,

            0.0

        )



    d["P_elz_bat_mw"] = alloc_bat(d["P_elz_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])

    d["P_nh3_bat_mw"] = alloc_bat(d["P_nh3_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])

    d["P_meoh_bat_mw"] = alloc_bat(d["P_meoh_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])

    d["P_crk_bat_mw"] = alloc_bat(d["P_crk_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])



    d["P_elz_grid_exact_mw"] = d["P_elz_ng_mw"] - d["P_elz_bat_mw"]

    d["P_nh3_grid_exact_mw"] = d["P_nh3_ng_mw"] - d["P_nh3_bat_mw"]

    d["P_meoh_grid_exact_mw"] = d["P_meoh_ng_mw"] - d["P_meoh_bat_mw"]

    d["P_crk_grid_exact_mw"] = d["P_crk_ng_mw"] - d["P_crk_bat_mw"]



    for col in ["P_elz_grid_exact_mw", "P_nh3_grid_exact_mw", "P_meoh_grid_exact_mw", "P_crk_grid_exact_mw"]:

        d[col] = clip_small_negative(d[col], tol=tol)



    d["grid_unit_sum_exact_mw"] = (

        d["P_elz_grid_exact_mw"] +

        d["P_nh3_grid_exact_mw"] +

        d["P_meoh_grid_exact_mw"] +

        d["P_crk_grid_exact_mw"]

    )

    d["bat_unit_sum_exact_mw"] = (

        d["P_elz_bat_mw"] +

        d["P_nh3_bat_mw"] +

        d["P_meoh_bat_mw"] +

        d["P_crk_bat_mw"]

    )



    d["grid_balance_gap_mw"] = d["grid_unit_sum_exact_mw"] - d["grid_import_mw"]

    d["bat_balance_gap_mw"] = d["bat_unit_sum_exact_mw"] - d["bat_dis_mw"]



    diag = pd.DataFrame({

        "metric": [

            "max_abs_grid_balance_gap_mw",

            "max_abs_battery_balance_gap_mw",

            "mean_abs_grid_balance_gap_mw",

            "mean_abs_battery_balance_gap_mw",

        ],

        "value": [

            float(d["grid_balance_gap_mw"].abs().max()),

            float(d["bat_balance_gap_mw"].abs().max()),

            float(d["grid_balance_gap_mw"].abs().mean()),

            float(d["bat_balance_gap_mw"].abs().mean()),

        ]

    })



    return d, diag

def reconstruct_intermediates(df: pd.DataFrame):

    d = df.copy()



    for p in ["H2", "NH3", "MeOH"]:

        for tag in ["g", "ng"]:

            d[f"tank_{p}_{tag}_next_kg"] = d[f"tank_{p}_{tag}_kg"].shift(-1)

            d[f"delta_tank_{p}_{tag}_kg"] = d[f"tank_{p}_{tag}_next_kg"] - d[f"tank_{p}_{tag}_kg"]



    d["has_next_state"] = d["tank_H2_g_next_kg"].notna()



    d["H_to_NH3_g_kgph"]   = M_H2_TO_NH3  * d["NH3_prod_g_kgph"]

    d["H_to_NH3_ng_kgph"]  = M_H2_TO_NH3  * d["NH3_prod_ng_kgph"]

    d["H_to_MeOH_g_kgph"]  = M_H2_TO_MEOH * d["MeOH_prod_g_kgph"]

    d["H_to_MeOH_ng_kgph"] = M_H2_TO_MEOH * d["MeOH_prod_ng_kgph"]



    denom = 1.0 - K_NH3_REC

    d["Q_crk_in_g_recon_kgph"] = (

        d["NH3_prod_g_kgph"] - d["NH3_sell_g_kgph"] - d["delta_tank_NH3_g_kg"]

    ) / denom

    d["Q_crk_in_ng_recon_kgph"] = (

        d["NH3_prod_ng_kgph"] - d["NH3_sell_ng_kgph"] - d["delta_tank_NH3_ng_kg"]

    ) / denom



    d["Q_crk_in_g_recon_kgph"] = clip_small_negative(d["Q_crk_in_g_recon_kgph"])

    d["Q_crk_in_ng_recon_kgph"] = clip_small_negative(d["Q_crk_in_ng_recon_kgph"])



    d["Q_crk_in_recon_sum_kgph"] = d["Q_crk_in_g_recon_kgph"] + d["Q_crk_in_ng_recon_kgph"]

    d["Q_crk_total_gap_kgph"] = d["Q_crk_in_recon_sum_kgph"] - d["crk_nh3_in_kgph"]



    d["Q_crk_split_valid"] = (

        d["has_next_state"] &

        d["Q_crk_in_g_recon_kgph"].notna() &

        d["Q_crk_in_ng_recon_kgph"].notna() &

        (d["Q_crk_total_gap_kgph"].abs() <= 5e-4)

    )



    valid = d["Q_crk_split_valid"] & (d["Q_crk_in_recon_sum_kgph"] > TOL)

    scale = np.where(valid, d["crk_nh3_in_kgph"] / d["Q_crk_in_recon_sum_kgph"], 1.0)

    d.loc[valid, "Q_crk_in_g_recon_kgph"] *= scale[valid]

    d.loc[valid, "Q_crk_in_ng_recon_kgph"] *= scale[valid]



    d["NH3_rec_g_recon_kgph"] = K_NH3_REC * d["Q_crk_in_g_recon_kgph"]

    d["NH3_rec_ng_recon_kgph"] = K_NH3_REC * d["Q_crk_in_ng_recon_kgph"]



    d["H_crk_out_g_recon_kgph"] = K_H2_OUT * d["Q_crk_in_g_recon_kgph"]

    d["H_crk_out_ng_recon_kgph"] = K_H2_OUT * d["Q_crk_in_ng_recon_kgph"]



    d["H_crk_fuel_g_recon_kgph"] = K_H2_FUEL * d["Q_crk_in_g_recon_kgph"]

    d["H_crk_fuel_ng_recon_kgph"] = K_H2_FUEL * d["Q_crk_in_ng_recon_kgph"]



    d["H_crk_out_total_gap_kgph"] = (

        d["H_crk_out_g_recon_kgph"] + d["H_crk_out_ng_recon_kgph"] - d["crk_h2_out_kgph"]

    )

    d["H_crk_fuel_total_gap_kgph"] = (

        d["H_crk_fuel_g_recon_kgph"] + d["H_crk_fuel_ng_recon_kgph"] - d["crk_h2_fuel_kgph"]

    )



    diag = pd.DataFrame({

        "metric": [

            "n_rows_total",

            "n_rows_with_next_state",

            "n_rows_valid_cracker_split",

            "n_rows_dropped_for_split_reconstruction",

            "max_abs_cracker_nh3in_gap_kgph",

            "max_abs_cracker_h2out_gap_kgph",

            "max_abs_cracker_h2fuel_gap_kgph",

        ],

        "value": [

            int(len(d)),

            int(d["has_next_state"].sum()),

            int(d["Q_crk_split_valid"].sum()),

            int((~d["Q_crk_split_valid"]).sum()),

            float(d["Q_crk_total_gap_kgph"].abs().max()),

            float(d["H_crk_out_total_gap_kgph"].abs().max()),

            float(d["H_crk_fuel_total_gap_kgph"].abs().max()),

        ]

    })



    return d, diag
# END unmodified original helpers

PRODUCTS = ('H2', 'NH3', 'MeOH')
LABELS = ('g', 'ng')
ORIGINS = ('T', 'U', 'D', 'N')  # T follows original scalar operation order.
CI_ATOL = 1e-9
CARBON_ATOL = 1e-6
RTOL = 1e-10
SIGN_CI_TOL = 1e-9
SIGN_GRID_TOL = 1e-6  # kgCO2 over one hour
EF_COL = 'grid_mef_upramp_pred_kg_per_kwh'


def carbon(value, origin):
    """Make total-plus-source vector without changing the original scalar value."""
    v = np.zeros(4)
    v[0] = value
    v[ORIGINS.index(origin)] = value
    return v


def corrected(v, stage, pool, timestamp, events):
    """Apply original -1 kgCO2 guard / max(total,0); book correction solely to N.

    U and D are never separately clipped or reclassified. Negative source tags,
    if present, remain visible; N is a signed numerical bookkeeping component.
    """
    if v[0] < -1.0:
        raise ValueError(f'Original numerical guard failed: {stage} {pool} {timestamp}: {v[0]}')
    v = v.copy()
    if v[0] < 0:
        adjustment = -v[0]
        events.append({'datetime_utc': timestamp, 'stage': stage, 'pool': pool,
                       'raw_total_kgco2': v[0], 'adjustment_kgco2': adjustment})
        v[0] = 0.0
        v[3] += adjustment
        return v, adjustment
    return v, 0.0


def replay_sources(d, reference):
    """One original-order replay with four parallel scalar carbon components.

    Six opening stocks; all withdrawals use opening CI. New production and
    returns enter closing stocks. Cracking-invalid hours follow the original
    branch (zero cracker transfers/fuel and no cracker electricity addition).
    """
    stocks = {(p,l): np.zeros(4) for p in PRODUCTS for l in LABELS}
    n = len(d)
    source_ci = np.zeros((n,3,2,4))
    pool_rows, global_rows, events = [], [], []
    max_ref_error = max_tag_error = 0.0
    for ti, r in enumerate(d.to_dict('records')):
        ts = r['datetime_utc']
        opening = {key:v.copy() for key,v in stocks.items()}
        ci = {key:(v / r[f'tank_{key[0]}_{key[1]}_kg']
                   if r[f'tank_{key[0]}_{key[1]}_kg'] > TOL else np.zeros(4))
              for key,v in opening.items()}
        for pi,p in enumerate(PRODUCTS):
            for li,l in enumerate(LABELS):
                source_ci[ti,pi,li] = ci[p,l]
                max_ref_error = max(max_ref_error, abs(ci[p,l][0]-reference.iloc[ti][f'CI_tank_{p}_{l}']))
        elec = {u:1000.0*r[f'P_{u}_grid_exact_mw']*r['ef_kg_per_kwh']
                for u in ('elz','nh3','meoh','crk')}
        external = {key:np.zeros(4) for key in stocks}
        external['H2','ng'] = carbon(elec['elz'],'U')
        for l in LABELS:
            external['NH3',l] = carbon(DE_NH3*r[f'NH3_prod_{l}_kgph'] + (elec['nh3'] if l=='ng' else 0.0),'D')
            external['MeOH',l] = carbon(DE_MEOH*r[f'MeOH_prod_{l}_kgph'] + (elec['meoh'] if l=='ng' else 0.0),'D')
        sales, fuel, pool_in, pool_out = {}, {}, {}, {}
        corrections = {key:0.0 for key in stocks}
        roundoff = {key:0.0 for key in stocks}
        for l in LABELS:
            h2n = ci['H2',l]*r[f'H_to_NH3_{l}_kgph']
            h2m = ci['H2',l]*r[f'H_to_MeOH_{l}_kgph']
            # Preserve original addition order: inherited + grid + direct.
            newn = h2n.copy()
            newm = h2m.copy()
            if l == 'ng':
                newn = newn + carbon(elec['nh3'],'D')
                newm = newm + carbon(elec['meoh'],'D')
            newn = newn + carbon(DE_NH3*r[f'NH3_prod_{l}_kgph'],'D')
            newm = newm + carbon(DE_MEOH*r[f'MeOH_prod_{l}_kgph'],'D')
            q = hf = 0.0
            ret = rec = feed = np.zeros(4)
            if bool(r['Q_crk_split_valid']):
                q = r[f'Q_crk_in_{l}_recon_kgph']
                hf = r[f'H_crk_fuel_{l}_recon_kgph']
                feed = ci['NH3',l]*q
                ret = ci['NH3',l]*r[f'NH3_rec_{l}_recon_kgph']
                rec = feed-ret
                if l == 'ng':
                    added = carbon(elec['crk'],'D')
                    rec = rec+added
                    external['H2',l] = external['H2',l]+added
                rec, adj = corrected(rec,'cracker_recovered_H2',f'H2_{l}',ts,events)
                corrections['H2',l] += adj
            for p in PRODUCTS:
                sales[p,l] = ci[p,l]*r[f'{p}_sell_{l}_kgph']
                fuel[p,l] = np.zeros(4)
            fuel['H2',l] = ci['H2',l]*hf
            # Scalar T follows the authoritative formula order exactly.
            hnew = (opening['H2',l]-sales['H2',l]-h2n-h2m
                    +carbon(elec['elz'] if l=='ng' else 0.0,'U')+rec-fuel['H2',l])
            nnew = opening['NH3',l]-sales['NH3',l]-ci['NH3',l]*q+newn+ret
            mnew = opening['MeOH',l]-sales['MeOH',l]+newm
            pool_in['H2',l] = carbon(elec['elz'] if l=='ng' else 0.0,'U')+rec
            pool_in['NH3',l] = newn+ret
            pool_in['MeOH',l] = newm
            pool_out['H2',l] = sales['H2',l]+h2n+h2m+fuel['H2',l]
            pool_out['NH3',l] = sales['NH3',l]+feed
            pool_out['MeOH',l] = sales['MeOH',l]
            for p,new in [('H2',hnew),('NH3',nnew),('MeOH',mnew)]:
                stocks[p,l], adj = corrected(new,'closing_stock',f'{p}_{l}',ts,events)
                corrections[p,l] += adj
                # Preserve T, U and D; reconcile floating summation solely in N.
                target_n = stocks[p,l][0] - (stocks[p,l][1] + stocks[p,l][2])
                floating = target_n - stocks[p,l][3]
                stocks[p,l][3] = target_n
                roundoff[p,l] = floating
                # Cracker correction is already in pool_in; only stock correction added here.
                pool_res = opening[p,l]+pool_in[p,l]-pool_out[p,l]+carbon(adj,'N')+np.array([0.,0.,0.,floating])-stocks[p,l]
                row = {'datetime_utc':ts,'product':p,'label':l,
                       'inventory_open_kg':r[f'tank_{p}_{l}_kg'],
                       'sold_kg':r[f'{p}_sell_{l}_kgph'],
                       'empty_pool':r[f'tank_{p}_{l}_kg']<=TOL,
                       'cracker_split_valid':bool(r['Q_crk_split_valid']),
                       'numerical_adjustment_kgco2':corrections[p,l],
                       'closing_stock_adjustment_kgco2':adj,
                       'cracker_transfer_adjustment_kgco2':corrections[p,l]-adj,
                       'source_roundoff_reconciliation_kgco2':floating,
                       'source_stock_closure_kgco2':stocks[p,l][0]-stocks[p,l][1:].sum()}
                for oi,o in enumerate(ORIGINS):
                    row.update({f'opening_{o}_kgco2':opening[p,l][oi],
                                f'closing_{o}_kgco2':stocks[p,l][oi],
                                f'inflow_{o}_kgco2':pool_in[p,l][oi],
                                f'outflow_{o}_kgco2':pool_out[p,l][oi],
                                f'sold_{o}_kgco2':sales[p,l][oi],
                                f'fuel_{o}_kgco2':fuel[p,l][oi],
                                f'pool_balance_residual_{o}_kgco2':pool_res[oi]})
                pool_rows.append(row)
                max_tag_error=max(max_tag_error,abs(row['source_stock_closure_kgco2']))
        op=np.sum(list(opening.values()),axis=0)
        cl=np.sum(list(stocks.values()),axis=0)
        ex=np.sum(list(external.values()),axis=0)
        so=np.sum(list(sales.values()),axis=0)
        fu=np.sum(list(fuel.values()),axis=0)
        nu=carbon(sum(corrections.values()),'N')
        nu[3] += sum(roundoff.values())
        resid=op+ex+nu-so-fu-cl
        gr={'datetime_utc':ts,'cracker_split_valid':bool(r['Q_crk_split_valid']),
            'unassigned_cracker_electricity_original_branch_kgco2':elec['crk'] if not bool(r['Q_crk_split_valid']) else 0.0}
        for oi,o in enumerate(ORIGINS):
            for name,val in [('opening',op),('closing',cl),('external',ex),('numerical',nu),('sold',so),('fuel',fu),('balance_residual',resid)]:
                gr[f'{name}_{o}_kgco2']=val[oi]
        global_rows.append(gr)
    pool=pd.DataFrame(pool_rows)
    gl=pd.DataFrame(global_rows)
    ev=pd.DataFrame(events,columns=['datetime_utc','stage','pool','raw_total_kgco2','adjustment_kgco2'])
    # This is a targeted extension check, not a rerun of published annual analyses.
    ref_pool=reference[[f'CI_tank_{p}_{l}' for p in PRODUCTS for l in LABELS]].to_numpy().reshape(n,3,2)
    np.testing.assert_allclose(source_ci[:,:,:,0],ref_pool,atol=CI_ATOL,rtol=RTOL,err_msg='Source extension differs from freshly calculated original pool CI')
    np.testing.assert_allclose(source_ci[:,:,:,1:].sum(3),source_ci[:,:,:,0],atol=CI_ATOL,rtol=RTOL)
    if max_tag_error > CARBON_ATOL or gl.filter(like='balance_residual').abs().max().max()>CARBON_ATOL:
        raise AssertionError('Tagged carbon conservation failed; inspect source ledger')
    return source_ci,pool,gl,ev,{'max_original_pool_CI_difference':max_ref_error,'max_source_stock_closure_kgco2':max_tag_error}


def find_file(base, explicit, candidates, label):
    if explicit:
        p=Path(explicit)
        if not p.exists(): raise FileNotFoundError(p)
        return p
    found=[base/c for c in candidates if (base/c).exists()]
    if len(found)!=1:
        raise ValueError(f'{label}: expected one candidate; found {found}. Set explicit --{label} path.')
    return found[0]


def check_year(d, name):
    expected=pd.date_range('2024-01-01',periods=8784,freq='h',tz='UTC')
    if not pd.DatetimeIndex(d['datetime_utc']).equals(expected):
        raise ValueError(f'{name} must contain each 2024 UTC hour exactly once, in order')


def derive_zero_initial_cis(recon_df: pd.DataFrame, signal_df: pd.DataFrame, signal_name: str):

    """

    Simplified assumption:

    all initial tank inventories at the start of the horizon are assigned zero carbon intensity.

    """

    return {

        "initial_inventory_rule_note": (

            "All initial tank inventories at the start of the study horizon are assigned zero carbon intensity."

        ),

        "initial_inventory_basis_signal": signal_name,

        "initial_inventory_basis_datetime_utc": "assumed_zero_initial_inventory",

        "initial_inventory_basis_day_of_year": 1,

        "initial_inventory_basis_hour_in_day": 0,

        "initial_inventory_basis_ef_kg_per_kwh": 0.0,

        "CI_init_H2_g_kgco2_perkg": 0.0,

        "CI_init_H2_ng_kgco2_perkg": 0.0,

        "CI_init_NH3_g_kgco2_perkg": 0.0,

        "CI_init_NH3_ng_kgco2_perkg": 0.0,

        "CI_init_MeOH_g_kgco2_perkg": 0.0,

        "CI_init_MeOH_ng_kgco2_perkg": 0.0,

    }

# -----------------------------

# STEP 2 — DYNAMIC CARBON LEDGER

# -----------------------------

def run_carbon_ledger(recon_df: pd.DataFrame, signal_name: str, signal_df: pd.DataFrame, init_ci_dict: dict):

    d = recon_df.merge(signal_df, on="datetime_utc", how="left").copy()

    if d["ef_kg_per_kwh"].isna().any():

        raise ValueError(f"Missing EF values after merge for {signal_name}")



    d["elec_carbon_elz_kgph"]  = 1000.0 * d["P_elz_grid_exact_mw"]  * d["ef_kg_per_kwh"]

    d["elec_carbon_nh3_kgph"]  = 1000.0 * d["P_nh3_grid_exact_mw"]  * d["ef_kg_per_kwh"]

    d["elec_carbon_meoh_kgph"] = 1000.0 * d["P_meoh_grid_exact_mw"] * d["ef_kg_per_kwh"]

    d["elec_carbon_crk_kgph"]  = 1000.0 * d["P_crk_grid_exact_mw"]  * d["ef_kg_per_kwh"]







    d["C_new_H2_g_kgco2"] = 0.0

    d["C_new_H2_ng_kgco2"] = d["elec_carbon_elz_kgph"]



    d["CI_new_H2_g_batch"] = np.where(d["H2_prod_g_kgph"] > TOL, d["C_new_H2_g_kgco2"] / d["H2_prod_g_kgph"], np.nan)

    d["CI_new_H2_ng_batch"] = np.where(d["H2_prod_ng_kgph"] > TOL, d["C_new_H2_ng_kgco2"] / d["H2_prod_ng_kgph"], np.nan)



    CH2g = float(d.loc[0, "tank_H2_g_kg"]   * init_ci_dict["CI_init_H2_g_kgco2_perkg"])

    CH2ng = float(d.loc[0, "tank_H2_ng_kg"] * init_ci_dict["CI_init_H2_ng_kgco2_perkg"])

    CNH3g = float(d.loc[0, "tank_NH3_g_kg"]   * init_ci_dict["CI_init_NH3_g_kgco2_perkg"])

    CNH3ng = float(d.loc[0, "tank_NH3_ng_kg"] * init_ci_dict["CI_init_NH3_ng_kgco2_perkg"])

    CMeOHg = float(d.loc[0, "tank_MeOH_g_kg"]   * init_ci_dict["CI_init_MeOH_g_kgco2_perkg"])

    CMeOHng = float(d.loc[0, "tank_MeOH_ng_kg"] * init_ci_dict["CI_init_MeOH_ng_kgco2_perkg"])



    rows = []



    for _, r in d.iterrows():

        ci_H2_g = CH2g / r["tank_H2_g_kg"] if r["tank_H2_g_kg"] > TOL else 0.0

        ci_H2_ng = CH2ng / r["tank_H2_ng_kg"] if r["tank_H2_ng_kg"] > TOL else 0.0



        ci_NH3_g = CNH3g / r["tank_NH3_g_kg"] if r["tank_NH3_g_kg"] > TOL else 0.0

        ci_NH3_ng = CNH3ng / r["tank_NH3_ng_kg"] if r["tank_NH3_ng_kg"] > TOL else 0.0



        ci_MeOH_g = CMeOHg / r["tank_MeOH_g_kg"] if r["tank_MeOH_g_kg"] > TOL else 0.0

        ci_MeOH_ng = CMeOHng / r["tank_MeOH_ng_kg"] if r["tank_MeOH_ng_kg"] > TOL else 0.0



        C_H2_to_NH3_g = ci_H2_g * r["H_to_NH3_g_kgph"]

        C_H2_to_NH3_ng = ci_H2_ng * r["H_to_NH3_ng_kgph"]

        C_H2_to_MeOH_g = ci_H2_g * r["H_to_MeOH_g_kgph"]

        C_H2_to_MeOH_ng = ci_H2_ng * r["H_to_MeOH_ng_kgph"]



        C_direct_NH3_g = DE_NH3 * r["NH3_prod_g_kgph"]

        C_direct_NH3_ng = DE_NH3 * r["NH3_prod_ng_kgph"]

        C_direct_MeOH_g = DE_MEOH * r["MeOH_prod_g_kgph"]

        C_direct_MeOH_ng = DE_MEOH * r["MeOH_prod_ng_kgph"]



        # Green downstream products inherit carbon from green H2 and direct process emissions only.

        # Grid-electricity carbon is assigned only to the non-green production stream.

        # Battery-fed non-green production remains non-green for attribution but receives zero grid CO2,

        # because elec_carbon_* is based only on reconstructed grid-fed unit electricity.



        C_new_NH3_g = (

            C_H2_to_NH3_g

            + C_direct_NH3_g

        )



        C_new_NH3_ng = (

            C_H2_to_NH3_ng

            + r["elec_carbon_nh3_kgph"]

            + C_direct_NH3_ng

        )



        C_new_MeOH_g = (

            C_H2_to_MeOH_g

            + C_direct_MeOH_g

        )



        C_new_MeOH_ng = (

            C_H2_to_MeOH_ng

            + r["elec_carbon_meoh_kgph"]

            + C_direct_MeOH_ng

        )



        if bool(r["Q_crk_split_valid"]):

            Qg = r["Q_crk_in_g_recon_kgph"]

            Qng = r["Q_crk_in_ng_recon_kgph"]



            NH3_rec_g = r["NH3_rec_g_recon_kgph"]

            NH3_rec_ng = r["NH3_rec_ng_recon_kgph"]



            H_out_g = r["H_crk_out_g_recon_kgph"]

            H_out_ng = r["H_crk_out_ng_recon_kgph"]



            H_fuel_g = r["H_crk_fuel_g_recon_kgph"]

            H_fuel_ng = r["H_crk_fuel_ng_recon_kgph"]



            # Cracker carbon propagation:

            # Incoming NH3 carries the current CI of the corresponding labeled NH3 tank.

            # Recovered NH3 returns with inherited NH3 carbon.

            # The remaining inherited carbon is assigned to recovered H2.

            # Cracker grid-electricity carbon is assigned only to the non-green recovered H2 stream.



            C_in_crk_NH3_g = ci_NH3_g * Qg

            C_in_crk_NH3_ng = ci_NH3_ng * Qng



            C_rec_NH3_g = ci_NH3_g * NH3_rec_g

            C_rec_NH3_ng = ci_NH3_ng * NH3_rec_ng



            C_H2_from_crk_g = C_in_crk_NH3_g - C_rec_NH3_g

            C_H2_from_crk_ng = C_in_crk_NH3_ng - C_rec_NH3_ng + r["elec_carbon_crk_kgph"]



            # Numerical safety for reconstructed cracker carbon flows.

            # Small negative values can occur from floating-point noise and reconstructed cracker splits.

            # Values below -1 kgCO2/h should be inspected because they may indicate a real reconstruction issue.



            CRK_CARBON_TOL = 1.0  # kgCO2/h



            if C_H2_from_crk_g < -CRK_CARBON_TOL:

                raise ValueError(

                    f"Large negative green cracker H2 carbon at {r['datetime_utc']}: {C_H2_from_crk_g}"

                )



            if C_H2_from_crk_ng < -CRK_CARBON_TOL:

                raise ValueError(

                    f"Large negative non-green cracker H2 carbon at {r['datetime_utc']}: {C_H2_from_crk_ng}"

                )



            C_H2_from_crk_g = max(C_H2_from_crk_g, 0.0)

            C_H2_from_crk_ng = max(C_H2_from_crk_ng, 0.0)

        else:

            Qg = Qng = 0.0

            H_fuel_g = H_fuel_ng = 0.0

            C_rec_NH3_g = C_rec_NH3_ng = 0.0

            C_H2_from_crk_g = C_H2_from_crk_ng = 0.0



        C_sell_H2_g = ci_H2_g * r["H2_sell_g_kgph"]

        C_sell_H2_ng = ci_H2_ng * r["H2_sell_ng_kgph"]



        C_sell_NH3_g = ci_NH3_g * r["NH3_sell_g_kgph"]

        C_sell_NH3_ng = ci_NH3_ng * r["NH3_sell_ng_kgph"]



        C_sell_MeOH_g = ci_MeOH_g * r["MeOH_sell_g_kgph"]

        C_sell_MeOH_ng = ci_MeOH_ng * r["MeOH_sell_ng_kgph"]



        def checked_nonnegative(x, name, timestamp, tol=1.0):

            """

            Clip small negative carbon stocks caused by numerical noise or reconstructed flows.

            Raise an error only for large negative values.

            tol is in kgCO2.

            """

            if x < -tol:

                raise ValueError(f"Large negative carbon stock in {name} at {timestamp}: {x}")

            return max(x, 0.0)



        CH2g_new = (

            CH2g

            - C_sell_H2_g

            - C_H2_to_NH3_g

            - C_H2_to_MeOH_g

            + r["C_new_H2_g_kgco2"]

            + C_H2_from_crk_g

            - ci_H2_g * H_fuel_g

        )



        CH2ng_new = (

            CH2ng

            - C_sell_H2_ng

            - C_H2_to_NH3_ng

            - C_H2_to_MeOH_ng

            + r["C_new_H2_ng_kgco2"]

            + C_H2_from_crk_ng

            - ci_H2_ng * H_fuel_ng

        )



        CNH3g_new = (

            CNH3g

            - C_sell_NH3_g

            - ci_NH3_g * Qg

            + C_new_NH3_g

            + C_rec_NH3_g

        )



        CNH3ng_new = (

            CNH3ng

            - C_sell_NH3_ng

            - ci_NH3_ng * Qng

            + C_new_NH3_ng

            + C_rec_NH3_ng

        )



        CMeOHg_new = (

            CMeOHg

            - C_sell_MeOH_g

            + C_new_MeOH_g

        )



        CMeOHng_new = (

            CMeOHng

            - C_sell_MeOH_ng

            + C_new_MeOH_ng

        )



        CH2g = checked_nonnegative(CH2g_new, "H2_g", r["datetime_utc"])

        CH2ng = checked_nonnegative(CH2ng_new, "H2_ng", r["datetime_utc"])

        CNH3g = checked_nonnegative(CNH3g_new, "NH3_g", r["datetime_utc"])

        CNH3ng = checked_nonnegative(CNH3ng_new, "NH3_ng", r["datetime_utc"])

        CMeOHg = checked_nonnegative(CMeOHg_new, "MeOH_g", r["datetime_utc"])

        CMeOHng = checked_nonnegative(CMeOHng_new, "MeOH_ng", r["datetime_utc"])



        rows.append({

            "datetime_utc": r["datetime_utc"],

            "day_of_year": int(r["day_of_year"]),

            "hour_in_day": int(r["hour_in_day"]),

            "CI_tank_H2_g": ci_H2_g,

            "CI_tank_H2_ng": ci_H2_ng,

            "CI_tank_NH3_g": ci_NH3_g,

            "CI_tank_NH3_ng": ci_NH3_ng,

            "CI_tank_MeOH_g": ci_MeOH_g,

            "CI_tank_MeOH_ng": ci_MeOH_ng,

            "C_sell_H2_g_kgco2": C_sell_H2_g,

            "C_sell_H2_ng_kgco2": C_sell_H2_ng,

            "C_sell_NH3_g_kgco2": C_sell_NH3_g,

            "C_sell_NH3_ng_kgco2": C_sell_NH3_ng,

            "C_sell_MeOH_g_kgco2": C_sell_MeOH_g,

            "C_sell_MeOH_ng_kgco2": C_sell_MeOH_ng,

            "H2_sold_total_kgph": float(r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"]),

            "NH3_sold_total_kgph": float(r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"]),

            "MeOH_sold_total_kgph": float(r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"]),

            "H2_sold_CI_kgco2_perkg_hourly": float((C_sell_H2_g + C_sell_H2_ng) / max(r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"], TOL)) if (r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"]) > TOL else np.nan,

            "NH3_sold_CI_kgco2_perkg_hourly": float((C_sell_NH3_g + C_sell_NH3_ng) / max(r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"], TOL)) if (r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"]) > TOL else np.nan,

            "MeOH_sold_CI_kgco2_perkg_hourly": float((C_sell_MeOH_g + C_sell_MeOH_ng) / max(r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"], TOL)) if (r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"]) > TOL else np.nan,

            "elec_carbon_elz_kgph": r["elec_carbon_elz_kgph"],

            "elec_carbon_nh3_kgph": r["elec_carbon_nh3_kgph"],

            "elec_carbon_meoh_kgph": r["elec_carbon_meoh_kgph"],

            "elec_carbon_crk_kgph": r["elec_carbon_crk_kgph"],

            "Q_crk_split_valid": bool(r["Q_crk_split_valid"]),

        })



    hourly = pd.DataFrame(rows)



    # Final-hour fallback: if the last hour has any unresolved value due to missing forward state,

    # use the hour-before-last values so the full study horizon remains covered.

    if len(hourly) >= 2:

        last_idx = hourly.index[-1]

        prev_idx = hourly.index[-2]

        fill_cols = [

            "CI_tank_H2_g", "CI_tank_H2_ng",

            "CI_tank_NH3_g", "CI_tank_NH3_ng",

            "CI_tank_MeOH_g", "CI_tank_MeOH_ng",

            "C_sell_H2_g_kgco2", "C_sell_H2_ng_kgco2",

            "C_sell_NH3_g_kgco2", "C_sell_NH3_ng_kgco2",

            "C_sell_MeOH_g_kgco2", "C_sell_MeOH_ng_kgco2",

        ]

        hourly.loc[last_idx, fill_cols] = hourly.loc[last_idx, fill_cols].where(hourly.loc[last_idx, fill_cols].notna(), hourly.loc[prev_idx, fill_cols])



    hourly["date"] = pd.to_datetime(hourly["datetime_utc"]).dt.date



    d2 = d.merge(

        hourly[[

            "datetime_utc",

            "CI_tank_H2_g", "CI_tank_H2_ng",

            "CI_tank_NH3_g", "CI_tank_NH3_ng",

            "CI_tank_MeOH_g", "CI_tank_MeOH_ng",

            "C_sell_H2_g_kgco2", "C_sell_H2_ng_kgco2",

            "C_sell_NH3_g_kgco2", "C_sell_NH3_ng_kgco2",

            "C_sell_MeOH_g_kgco2", "C_sell_MeOH_ng_kgco2"

        ]],

        on="datetime_utc",

        how="left"

    )



    daily = d2.groupby("day_of_year", as_index=False).agg(

        date=("datetime_utc", lambda x: str(pd.to_datetime(x.iloc[0]).date())),



        H2_sold_g_kg=("H2_sell_g_kgph", "sum"),

        H2_sold_ng_kg=("H2_sell_ng_kgph", "sum"),

        H2_carbon_g_kg=("C_sell_H2_g_kgco2", "sum"),

        H2_carbon_ng_kg=("C_sell_H2_ng_kgco2", "sum"),



        NH3_sold_g_kg=("NH3_sell_g_kgph", "sum"),

        NH3_sold_ng_kg=("NH3_sell_ng_kgph", "sum"),

        NH3_carbon_g_kg=("C_sell_NH3_g_kgco2", "sum"),

        NH3_carbon_ng_kg=("C_sell_NH3_ng_kgco2", "sum"),



        MeOH_sold_g_kg=("MeOH_sell_g_kgph", "sum"),

        MeOH_sold_ng_kg=("MeOH_sell_ng_kgph", "sum"),

        MeOH_carbon_g_kg=("C_sell_MeOH_g_kgco2", "sum"),

        MeOH_carbon_ng_kg=("C_sell_MeOH_ng_kgco2", "sum"),

    )



    for p in PRODUCTS:

        daily[f"{p}_sold_total_kg"] = daily[f"{p}_sold_g_kg"] + daily[f"{p}_sold_ng_kg"]

        daily[f"{p}_sold_green_share_pct"] = np.where(

            daily[f"{p}_sold_total_kg"] > TOL,

            100.0 * daily[f"{p}_sold_g_kg"] / daily[f"{p}_sold_total_kg"],

            np.nan

        )

        daily[f"{p}_sold_CI_kgco2_perkg"] = np.where(

            daily[f"{p}_sold_total_kg"] > TOL,

            (daily[f"{p}_carbon_g_kg"] + daily[f"{p}_carbon_ng_kg"]) / daily[f"{p}_sold_total_kg"],

            np.nan

        )



    summary = pd.DataFrame([{

        "initial_inventory_method_note": init_ci_dict["initial_inventory_rule_note"],

        "initial_inventory_basis_signal": init_ci_dict["initial_inventory_basis_signal"],

        "initial_inventory_basis_ef_kg_per_kwh": init_ci_dict["initial_inventory_basis_ef_kg_per_kwh"],

        "initial_inventory_basis_datetime_utc": init_ci_dict["initial_inventory_basis_datetime_utc"],

        "initial_inventory_basis_day_of_year": init_ci_dict["initial_inventory_basis_day_of_year"],

        "initial_inventory_basis_hour_in_day": init_ci_dict["initial_inventory_basis_hour_in_day"],



        "CI_init_H2_g_kgco2_perkg": init_ci_dict["CI_init_H2_g_kgco2_perkg"],

        "CI_init_H2_ng_kgco2_perkg": init_ci_dict["CI_init_H2_ng_kgco2_perkg"],

        "CI_init_NH3_g_kgco2_perkg": init_ci_dict["CI_init_NH3_g_kgco2_perkg"],

        "CI_init_NH3_ng_kgco2_perkg": init_ci_dict["CI_init_NH3_ng_kgco2_perkg"],

        "CI_init_MeOH_g_kgco2_perkg": init_ci_dict["CI_init_MeOH_g_kgco2_perkg"],

        "CI_init_MeOH_ng_kgco2_perkg": init_ci_dict["CI_init_MeOH_ng_kgco2_perkg"],



        "H2_sold_green_share_pct": float(100.0 * daily["H2_sold_g_kg"].sum() / max(daily["H2_sold_total_kg"].sum(), TOL)),

        "NH3_sold_green_share_pct": float(100.0 * daily["NH3_sold_g_kg"].sum() / max(daily["NH3_sold_total_kg"].sum(), TOL)),

        "MeOH_sold_green_share_pct": float(100.0 * daily["MeOH_sold_g_kg"].sum() / max(daily["MeOH_sold_total_kg"].sum(), TOL)),



        "H2_sold_CI_kgco2_perkg": float((daily["H2_carbon_g_kg"].sum() + daily["H2_carbon_ng_kg"].sum()) / max(daily["H2_sold_total_kg"].sum(), TOL)),

        "NH3_sold_CI_kgco2_perkg": float((daily["NH3_carbon_g_kg"].sum() + daily["NH3_carbon_ng_kg"].sum()) / max(daily["NH3_sold_total_kg"].sum(), TOL)),

        "MeOH_sold_CI_kgco2_perkg": float((daily["MeOH_carbon_g_kg"].sum() + daily["MeOH_carbon_ng_kg"].sum()) / max(daily["MeOH_sold_total_kg"].sum(), TOL)),



        "hours_excluded_from_propagation_due_to_missing_next_state": int((~d["has_next_state"]).sum()),

        "hours_excluded_from_cracker_split_due_to_invalid_reconstruction": int((~d["Q_crk_split_valid"]).sum()),

        "electricity_split_method_note": (

            "Exact sink-level grid power is reconstructed from total unit power, saved green share, "

            "and proportional allocation of battery discharge across non-green unit demand. "

            "Battery-fed power carries zero electricity carbon."

        ),

    }])



    return d2, hourly, daily, summary

def main():
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    paths = {'M0': Path(M0_FILE), 'M2': Path(M2_FILE), 'signal': Path(SIGNAL_FILE)}
    for label, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"{label} CSV not found: {path}. Upload it to Colab and edit the path at the top."
            )
    sig=load_signal_df(paths['signal'],EF_COL);check_year(sig,'signal')
    datasets={}; refs={}; tags={}; audits={}; poolall=[];globalall=[];eventsall=[]
    for model in ['M0','M2']:
        raw=load_model_df(paths[model]);check_year(raw,model)
        power,_=reconstruct_unit_grid_battery_split(raw)
        recon,_=reconstruct_intermediates(power)
        d=recon.merge(sig,on='datetime_utc',validate='one_to_one',how='left')
        init = derive_zero_initial_cis(recon, sig, 'UpRamp MEF')
        merged, ref, daily, annual = run_carbon_ledger(recon, 'UpRamp MEF', sig, init)
        check_year(ref, 'original ledger '+model)
        ref.to_csv(out/f'{model}_original_hourly_carbon_ledger.csv', index=False)
        daily.to_csv(out/f'{model}_original_daily_carbon_ledger.csv', index=False)
        annual.to_csv(out/f'{model}_original_annual_carbon_summary.csv', index=False)
        for u in ['elz','nh3','meoh','crk']:
            np.testing.assert_allclose(1000*d[f'P_{u}_grid_exact_mw']*d.ef_kg_per_kwh,
                                       ref[f'elec_carbon_{u}_kgph'],rtol=RTOL,atol=CARBON_ATOL)
        np.testing.assert_array_equal(d.Q_crk_split_valid,ref.Q_crk_split_valid)
        src,pool,gl,ev,audit=replay_sources(d,ref)
        for frame in [pool,gl,ev]: frame.insert(0,'model',model)
        poolall.append(pool);globalall.append(gl);eventsall.append(ev)
        datasets[model]=d;refs[model]=ref;tags[model]=src;audits[model]=audit
    pool=pd.concat(poolall,ignore_index=True);gl=pd.concat(globalall,ignore_index=True);ev=pd.concat(eventsall,ignore_index=True)
    rows=[]
    d0,d2=datasets['M0'],datasets['M2']
    for pi,p in enumerate(PRODUCTS):
        s={m:datasets[m][[f'{p}_sell_{l}_kgph' for l in LABELS]].to_numpy() for m in ['M0','M2']}
        q={m:s[m].sum(1) for m in s}
        w={m:np.divide(s[m],q[m][:,None],out=np.full_like(s[m],np.nan),where=q[m][:,None]>TOL) for m in s}
        c={m:tags[m][:,pi] for m in s} # hour,label,origin
        y={m:refs[m][f'{p}_sold_CI_kgco2_perkg_hourly'].to_numpy() for m in s}
        for m in s:
            np.testing.assert_allclose((w[m]*c[m][:,:,0]).sum(1),y[m],atol=CI_ATOL,rtol=RTOL,equal_nan=True)
        avgw=(w['M2']+w['M0'])/2
        components={o:(avgw*(c['M2'][:,:,oi]-c['M0'][:,:,oi])).sum(1) for oi,o in enumerate(ORIGINS) if o!='T'}
        compW=(((c['M2'][:,:,0]+c['M0'][:,:,0])/2)*(w['M2']-w['M0'])).sum(1)
        compI=(avgw*(c['M2'][:,:,0]-c['M0'][:,:,0])).sum(1)
        gap=y['M2']-y['M0']
        valid=(q['M0']>TOL)&(q['M2']>TOL)&np.isfinite(gap)
        grid={m:1000*datasets[m]['grid_import_mw'].to_numpy()*sig['ef_kg_per_kwh'].to_numpy() for m in s}
        gdiff=grid['M2']-grid['M0']
        f=pd.DataFrame({'datetime_utc':d0.datetime_utc,'product':p,
            'original_CI_M0_kgco2_perkg':y['M0'],'original_CI_M2_kgco2_perkg':y['M2'],
            'Psi_CI_gap_kgco2_perkg':gap,
            'A_U_kgco2_perkg':components['U'],'A_D_kgco2_perkg':components['D'],
            'A_W_kgco2_perkg':compW,'A_N_numerical_kgco2_perkg':components['N'],
            'A_I_total_inventory_kgco2_perkg':compI,
            'two_term_residual_kgco2_perkg':gap-compI-compW,
            'three_term_residual_kgco2_perkg':gap-components['U']-components['D']-compW,
            'corrected_residual_kgco2_perkg':gap-components['U']-components['D']-compW-components['N'],
            'valid_paired_withdrawal':valid,
            'higher_CI_strict':valid&(gap>0),'higher_CI':valid&(gap>0),
            'higher_CI_tolerance':valid&(gap>SIGN_CI_TOL),
            'grid_emission_gap_kgco2':gdiff,
            'lower_grid_emissions':gdiff < 0,
            'lower_grid_emissions_tolerance':gdiff < -SIGN_GRID_TOL,
            'higher_CI_and_lower_grid':valid&(gap>0)&(gdiff < 0),
            'higher_CI_and_lower_grid_tolerance':valid&(gap>SIGN_CI_TOL)&(gdiff < -SIGN_GRID_TOL),
            'near_zero_CI_gap':np.abs(gap)<=SIGN_CI_TOL,
            'empty_label_either_model':False,'positive_sale_from_empty_pool':False,
            'label_support_mismatch':False})
        for m in s:
            f[f'sold_mass_{m}_kg']=q[m]
            f[f'grid_emissions_{m}_kgco2']=grid[m]
            f[f'cracker_split_valid_{m}']=datasets[m]['Q_crk_split_valid'].to_numpy()
            for u in ['elz','nh3','meoh','crk']:
                f[f'power_{u}_{m}_mw']=datasets[m][f'P_{u}_mw'].to_numpy()
                f[f'grid_power_{u}_{m}_mw']=datasets[m][f'P_{u}_grid_exact_mw'].to_numpy()
                f[f'new_electricity_carbon_{u}_{m}_kgco2']=1000*datasets[m][f'P_{u}_grid_exact_mw'].to_numpy()*sig['ef_kg_per_kwh'].to_numpy()
            f[f'production_{m}_kg']=datasets[m][[f'{p}_prod_{l}_kgph' for l in LABELS]].sum(1).to_numpy()
            f[f'cracking_NH3_feed_{m}_kg']=datasets[m]['crk_nh3_in_kgph'].to_numpy()
            f[f'grid_import_{m}_mw']=datasets[m]['grid_import_mw'].to_numpy()
            f[f'inventory_total_{m}_kg']=datasets[m][[f'tank_{p}_{l}_kg' for l in LABELS]].sum(1).to_numpy()
            for li,l in enumerate(LABELS):
                f[f'sold_weight_{l}_{m}']=w[m][:,li]
                f[f'inventory_{l}_{m}_kg']=datasets[m][f'tank_{p}_{l}_kg'].to_numpy()
                for oi,o in enumerate(ORIGINS): f[f'pool_CI_{o}_{l}_{m}_kgco2_perkg']=c[m][:,li,oi]
                empty=datasets[m][f'tank_{p}_{l}_kg'].to_numpy()<=TOL
                f['empty_label_either_model'] |= empty
                f['positive_sale_from_empty_pool'] |= empty&(s[m][:,li]>TOL)
        for l in LABELS:
            f['label_support_mismatch'] |= ((d0[f'tank_{p}_{l}_kg'].to_numpy()>TOL)!=(d2[f'tank_{p}_{l}_kg'].to_numpy()>TOL))
        f['ef_kg_per_kwh']=sig['ef_kg_per_kwh'].to_numpy()
        f['sold_mass_gap_kg']=q['M2']-q['M0']
        f['green_withdrawal_weight_gap']=w['M2'][:,0]-w['M0'][:,0]
        f['inventory_mass_gap_kg']=f['inventory_total_M2_kg']-f['inventory_total_M0_kg']
        f['production_gap_kg']=f['production_M2_kg']-f['production_M0_kg']
        f['source_corrected_closure_valid']=np.abs(f.corrected_residual_kgco2_perkg)<=CI_ATOL
        f['three_term_closure_valid']=np.abs(f.three_term_residual_kgco2_perkg)<=CI_ATOL
        if valid.any() and not f.loc[valid,'source_corrected_closure_valid'].all():
            raise AssertionError('Decomposition does not close against calculated original sold CI')
        rows.append(f)
    hourly=pd.concat(rows,ignore_index=True).sort_values(['datetime_utc','product']).reset_index(drop=True)
    summaries=[];coverage=[]
    for p in PRODUCTS:
        f=hourly[hourly['product']==p]
        coverage.append({'product':p,'paired_valid_hours':int(f.valid_paired_withdrawal.sum()),
                         'total_hours':len(f),'empty_label_hours':int(f.empty_label_either_model.sum()),
                         'support_mismatch_hours':int(f.label_support_mismatch.sum()),
                         'positive_sale_from_empty_pool_hours':int(f.positive_sale_from_empty_pool.sum()),
                         'higher_CI_strict_hours':int(f.higher_CI_strict.sum()),
                         'higher_CI_tolerance_hours':int(f.higher_CI_tolerance.sum()),
                         'positive_CI_and_lower_grid_strict_hours':int(f.higher_CI_and_lower_grid.sum()),
                         'positive_CI_and_lower_grid_tolerance_hours':int(f.higher_CI_and_lower_grid_tolerance.sum()),
                         'max_abs_sold_mass_gap_kg':float(f.sold_mass_gap_kg.abs().max())})
        for group,mask in [('higher_CI',f.higher_CI),('higher_CI_and_lower_grid',f.higher_CI_and_lower_grid),
                           ('higher_CI_tolerance',f.higher_CI_tolerance),('higher_CI_and_lower_grid_tolerance',f.higher_CI_and_lower_grid_tolerance)]:
            g=f[mask]
            row={'product':p,'subset':group,'n_valid_hours':len(g),
                 'mean_Psi_CI_gap_kgco2_perkg':g.Psi_CI_gap_kgco2_perkg.mean(),
                 'mean_A_I_total_inventory_kgco2_perkg':g.A_I_total_inventory_kgco2_perkg.mean(),
                 'empty_label_hours':int(g.empty_label_either_model.sum()),
                 'support_mismatch_hours':int(g.label_support_mismatch.sum())}
            for term in ['U','D','W','N_numerical']:
                x=g[f'A_{term}_kgco2_perkg']
                row[f'mean_A_{term}_kgco2_perkg']=x.mean()
                row[f'p05_A_{term}_kgco2_perkg']=x.quantile(.05)
                row[f'p95_A_{term}_kgco2_perkg']=x.quantile(.95)
                row[f'mean_positive_A_{term}_kgco2_perkg']=x.clip(lower=0).mean()
                row[f'mean_negative_A_{term}_kgco2_perkg']=x.clip(upper=0).mean()
            row['max_abs_three_term_residual_kgco2_perkg']=g.three_term_residual_kgco2_perkg.abs().max()
            row['max_abs_corrected_residual_kgco2_perkg']=g.corrected_residual_kgco2_perkg.abs().max()
            summaries.append(row)
    summary=pd.DataFrame(summaries)
    # Write audit files even if strict three-term closure fails; do not hide N.
    hourly.to_csv(out/'hourly_source_origin_decomposition.csv',index=False)
    compact=['product','subset','n_valid_hours','mean_Psi_CI_gap_kgco2_perkg',
             'mean_A_U_kgco2_perkg','mean_A_D_kgco2_perkg','mean_A_W_kgco2_perkg',
             'mean_A_N_numerical_kgco2_perkg','max_abs_corrected_residual_kgco2_perkg']
    summary.loc[~summary['subset'].str.endswith('_tolerance'),compact].to_csv(out/'SI_source_origin_summary.csv',index=False)
    summary.loc[summary['subset'].str.endswith('_tolerance'),compact].to_csv(out/'sign_tolerance_summary.csv',index=False)
    summary.to_csv(out/'contribution_distribution_statistics.csv',index=False)
    support=[]
    for p in PRODUCTS:
        f=hourly[hourly['product']==p]
        for subset in ['higher_CI','higher_CI_and_lower_grid']:
            for restriction,mask in [('all',f[subset]),('common_positive_label_support',f[subset]&~f.empty_label_either_model)]:
                g=f[mask]
                rec={'product':p,'subset':subset,'support':restriction,'hours':len(g)}
                for col in ['Psi_CI_gap_kgco2_perkg','A_U_kgco2_perkg','A_D_kgco2_perkg','A_W_kgco2_perkg','A_N_numerical_kgco2_perkg']:
                    rec['mean_'+col]=g[col].mean()
                support.append(rec)
    pd.DataFrame(support).to_csv(out/'support_diagnostics.csv',index=False)
    # Every run includes machine-readable descriptions of the hourly columns.
    descriptions={
        'datetime_utc':'UTC start of the original one-hour flow/withdrawal interval.',
        'product':'Product identifier H2, NH3 or MeOH.',
        'Psi_CI_gap_kgco2_perkg':'Freshly calculated original sold CI M2 minus M0; absolute gap, denoted Psi in the report.',
        'A_U_kgco2_perkg':'Symmetric electrolysis-origin inventory-intensity contribution.',
        'A_D_kgco2_perkg':'Symmetric downstream-origin inventory-intensity contribution.',
        'A_W_kgco2_perkg':'Symmetric labeled withdrawal-composition contribution, using total pool CI.',
        'A_N_numerical_kgco2_perkg':'Explicit symmetric numerical-adjustment inventory-intensity contribution; not a physical carbon origin.',
        'A_I_total_inventory_kgco2_perkg':'Exact two-term fallback inventory-intensity contribution, using original total pool CI.',
        'three_term_residual_kgco2_perkg':'Psi minus A_U minus A_D minus A_W; includes A_N, not generally zero.',
        'corrected_residual_kgco2_perkg':'Psi minus A_U minus A_D minus A_W minus A_N.',
        'two_term_residual_kgco2_perkg':'Psi minus A_I minus A_W.',
        'valid_paired_withdrawal':'Both calculated sold masses exceed original TOL=1e-6 kg and both calculated CIs are finite.',
        'higher_CI':'Valid pair and calculated Psi strictly greater than zero (no sign tolerance).',
        'higher_CI_strict':'Same as higher_CI; retained explicit sign-definition flag.',
        'higher_CI_tolerance':'Valid pair and Psi greater than 1e-9 kgCO2/kg.',
        'lower_grid_emissions':'M2 minus M0 hourly grid emissions strictly less than zero.',
        'lower_grid_emissions_tolerance':'M2 minus M0 hourly grid emissions below -1e-6 kgCO2.',
        'higher_CI_and_lower_grid':'Intersection of strict higher-CI and strict lower-grid-emission flags.',
        'higher_CI_and_lower_grid_tolerance':'Intersection of the two tolerance-based sign flags.',
        'near_zero_CI_gap':'Absolute Psi at most 1e-9 kgCO2/kg; diagnostic, not an exclusion from strict summary.',
        'empty_label_either_model':'At least one labeled opening pool mass at most original TOL=1e-6 kg.',
        'positive_sale_from_empty_pool':'Any labeled sale above TOL from a pool with opening mass at most TOL.',
        'label_support_mismatch':'An opening label is above TOL in only one of the two cases.',
        'source_corrected_closure_valid':'Absolute corrected residual at most 1e-9 kgCO2/kg.',
        'three_term_closure_valid':'Absolute three-term residual at most 1e-9 kgCO2/kg.',
        'ef_kg_per_kwh':'Common forecast-based UpRamp MEF evaluation signal, unchanged.'}
    dictionary=[]
    for col in hourly.columns:
        description=descriptions.get(col)
        if description is None:
            if col.startswith('original_CI_'): description='Freshly calculated original hourly sold-product CI for indicated model, kgCO2/kg.'
            elif col.startswith('pool_CI_'): description='Opening labeled pool carbon intensity, indicated model/label/origin (T total, U electrolysis, D downstream, N numerical), kgCO2/kg.'
            elif col.startswith('sold_weight_'): description='Labeled sold mass divided by total hourly sold mass for indicated model; dimensionless.'
            elif col.startswith('inventory_'): description='Opening inventory mass (labeled, total or M2-minus-M0 gap), kg; original trajectories unchanged.'
            elif col.startswith('new_electricity_carbon_'): description='Current-hour process electricity carbon from original reconstructed grid power times common signal; not the source intensity contribution.'
            elif col.startswith('grid_emission'): description='Hourly grid-import burden or M2-minus-M0 gap: 1000 times grid_import_mw times common EF, kgCO2.'
            elif col.startswith('grid_power_'): description='Process grid power from original reconstruction, MW.'
            elif col.startswith('power_'): description='Original total process power, MW.'
            elif col.startswith('grid_import_'): description='Original total grid import, MW.'
            elif col.startswith('production_'): description='Original product production (g+ng), or M2-minus-M0 gap, kg in the hour.'
            elif col.startswith('sold_mass_'): description='Original total sold product mass, or M2-minus-M0 gap, kg in the hour.'
            elif col.startswith('cracking_NH3_'): description='Original aggregate NH3 cracker feed, kg in the hour.'
            elif col.startswith('cracker_split_valid_'): description='Original labeled-cracker reconstruction validity; false invokes original zero-transfer branch.'
            elif col=='green_withdrawal_weight_gap': description='M2 minus M0 green-labeled fraction of hourly sold product.'
            else: raise ValueError('Missing data dictionary description: '+col)
        dictionary.append({'column':col,'description':description})
    pd.DataFrame(dictionary).to_csv(out/'hourly_data_dictionary.csv',index=False)
    pd.DataFrame(coverage).to_csv(out/'coverage.csv',index=False)
    pool.to_csv(out/'source_inventory_ledger.csv.gz',index=False,compression={'method':'gzip','mtime':0})
    gl.to_csv(out/'source_global_balance.csv',index=False)
    ev.to_csv(out/'numerical_corrections.csv',index=False)
    balances=[]
    for m in ['M0','M2']:
        gg=gl[gl.model==m]
        for o in ORIGINS:
            initial=float(gg[f'opening_{o}_kgco2'].iloc[0]);closing=float(gg[f'closing_{o}_kgco2'].iloc[-1])
            external=float(gg[f'external_{o}_kgco2'].sum());num=float(gg[f'numerical_{o}_kgco2'].sum())
            sold=float(gg[f'sold_{o}_kgco2'].sum());fuel=float(gg[f'fuel_{o}_kgco2'].sum())
            balances.append({'model':m,'origin':o,'initial_kgco2':initial,'external_kgco2':external,
                             'numerical_kgco2':num,'sold_kgco2':sold,'fuel_kgco2':fuel,'closing_kgco2':closing,
                             'annual_balance_residual_kgco2':initial+external+num-sold-fuel-closing,
                             'max_abs_hourly_balance_residual_kgco2':gg[f'balance_residual_{o}_kgco2'].abs().max()})
    pd.DataFrame(balances).to_csv(out/'source_balance_summary.csv',index=False)
    validation={'version':'1.1.0-standalone','python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                'models':audits,'CI_absolute_tolerance':CI_ATOL,'carbon_absolute_tolerance_kgco2':CARBON_ATOL,
                'relative_tolerance':RTOL,'sign_CI_tolerance':SIGN_CI_TOL,'sign_grid_tolerance_kgco2':SIGN_GRID_TOL,
                'source_corrected_decomposition_pass':bool(hourly.loc[hourly.valid_paired_withdrawal,'source_corrected_closure_valid'].all()),
                'strict_three_term_pass_at_tolerance':bool(hourly.loc[hourly.valid_paired_withdrawal,'three_term_closure_valid'].all()),
                'max_abs_three_term_residual':float(hourly.three_term_residual_kgco2_perkg.abs().max()),
                'max_abs_corrected_residual':float(hourly.corrected_residual_kgco2_perkg.abs().max()),
                'max_abs_two_term_residual':float(hourly.two_term_residual_kgco2_perkg.abs().max()),
                'numerical_correction_count':len(ev),'numerical_correction_total_kgco2':float(ev.adjustment_kgco2.sum()),
                'absolute_source_roundoff_reconciliation_kgco2':float(pool.source_roundoff_reconciliation_kgco2.abs().sum()),
                'min_U_stock_kgco2':float(pool.closing_U_kgco2.min()),'min_D_stock_kgco2':float(pool.closing_D_kgco2.min()),
                'max_abs_pool_balance_residual_kgco2':float(pool.filter(like='pool_balance_residual').abs().max().max()),
                'coverage':coverage,
                'inputs':{k:{'filename':v.name,'sha256':hashlib.sha256(v.read_bytes()).hexdigest()} for k,v in paths.items()}}
    (out/'validation.json').write_text(json.dumps(validation,indent=2,allow_nan=False))
    print(summary[~summary['subset'].str.endswith('_tolerance')][['product','subset','n_valid_hours','mean_Psi_CI_gap_kgco2_perkg','mean_A_U_kgco2_perkg','mean_A_D_kgco2_perkg','mean_A_W_kgco2_perkg','mean_A_N_numerical_kgco2_perkg']].to_string(index=False))
    print('\nThree-term closure:',validation['strict_three_term_pass_at_tolerance'],
          '| Source-corrected closure:',validation['source_corrected_decomposition_pass'])
    print('Numerical corrections:',len(ev),'| Results:',out.resolve())
    zip_path = Path(ZIP_FILE)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for result_file in sorted(out.rglob("*")):
            if result_file.is_file() and result_file.resolve() != zip_path.resolve():
                archive.write(result_file, arcname=str(Path(out.name) / result_file.relative_to(out)))
    print("ZIP archive:", zip_path.resolve())
    return validation

if __name__ == '__main__':
    main()


product                   subset  n_valid_hours  mean_Psi_CI_gap_kgco2_perkg  mean_A_U_kgco2_perkg  mean_A_D_kgco2_perkg  mean_A_W_kgco2_perkg  mean_A_N_numerical_kgco2_perkg
     H2                higher_CI           1940                     0.479275             -0.036765             -0.000445              0.516574                   -8.848014e-05
     H2 higher_CI_and_lower_grid            810                     0.495614             -0.014743             -0.000388              0.510785                   -3.920771e-05
    NH3                higher_CI           3140                     0.178406              0.006764              0.002378              0.169264                    7.705029e-12
    NH3 higher_CI_and_lower_grid           1283                     0.187828              0.016883              0.002783              0.168162                    6.649777e-12
   MeOH                higher_CI           2915                     0.188048             -0.033529             -0.000684     